# Model Selection and Clustering

This notebook encapsulates the complete pipeline for selecting the large language models (LLMs) to be subjected to study.

The process consists of:
1. **Data Collection**: Fetching model details from predefined Hugging Face collections.
2. **Filtering**: Applying a set of rules to ignore specialized, quantized, reasoning/thinking, and MoE models.
3. **Prioritization**: Choosing instruction-tuned models over their base versions.
4. **Clustering**: Grouping models into sizes (Tiny, Small, Standard, Mid-range) using Jenks Natural Breaks.
5. **Selection**: Selecting the most recent or largest parameter model available within each category for each family.

### Setup & Imports
We import necessary libraries to handle API requests (`huggingface_hub`), data manipulation (`pandas`), and clustering (`jenkspy`).

In [13]:
import pandas as pd
import jenkspy
import re
from huggingface_hub import HfApi, get_collection

### 1. Configuration and Filtering Rules
Here we define our constants and parameters for model collection. We set maximum limits, regex patterns to extract parameter sizes, and lists of keywords/types to ignore (e.g., GGUF, MoE, Vision models) ensuring we only capture standard text generation base or instruct models.

In [14]:
# Maximum parameter count in billions (hardware constraints)
MAX_PARAMS = 50.0

# Regex Patterns
PARAM_PATTERN = re.compile(r'(\d+(?:\.\d+)?)\s*[Bb]', re.IGNORECASE)
M_PARAM_PATTERN = re.compile(r'(\d+(?:\.\d+)?)\s*m', re.IGNORECASE)
# Matches patterns like 30B-A3B or 14b-a2b (MoE architecture)
MOE_PATTERN = re.compile(r'-\d+B-A?\d+[B|E]', re.IGNORECASE)

# Instruction Suffixes
PRIORITIZE_SUFFIXES = ["-it", "-instruct", "-chat", "-instruction", "-hf"]

# Types to ignore
IGNORE_TYPES = {
    "audio-to-audio",
    "text-to-audio",
    "audio-text-to-audio",
    "text-to-image",
    "image-to-image",
    "image-text-to-image"
}

# Specific words to ignore in model names (case-insensitive)
IGNORE_WORDS = {
    # Quantized versions
    "gguf", "awq", "gptq", "quant", "quantized", "int4", "int8", "fp8", "4bit", "6bit", "8bit",
    # Specialized versions
    "special", "speciale", "vision", "image", "audio", "code", "coder", "math", "ocr", "guard",
    "pytorch", "keras", "flax", "cpp", "tflite", "mlx", "terminus", "exp",
    # Not instruction-tuned versions
    "base", "-pt",
    # Reasoning/Thinking models
    "reason", "reasoning", "think", "thinking",
    # Mixture-of-Experts architecture
    "moe",
    # Others
    "deprecated"
}

# Specific models to ignore by exact model ID
IGNORE_MODELS = {
    "meta-llama/Llama-2-7b-hf",
    "meta-llama/Llama-2-13b-hf"
}

### 2. Helper Functions
These functions parse model names to extract parameter sizes and query the Hugging Face API (`get_model_details`). During the API query, the model is checked against the ignoring rules before being accepted.

In [15]:
def format_to_billions(num):
    """Converts raw integer or float to 'X.XX' string."""
    if num is None or num == "N/A":
        return "N/A"
    try:
        return f"{float(num) / 1e9:.2f}"
    except (ValueError, TypeError):
        return str(num)

def extract_params_from_name(model_id):
    """Tries to find parameter count in the model name (e.g., '1.7B') and formats to two decimals."""
    match = PARAM_PATTERN.search(model_id.split('/')[-1])
    if match:
        try:
            return f"{float(match.group(1)):.2f}"
        except ValueError:
            return "N/A"
    else:
        alt_match = M_PARAM_PATTERN.search(model_id.split('/')[-1])
        if alt_match:
            try:
                return f"{(float(alt_match.group(1))/1000):.2f}"
            except ValueError:
                return "N/A"
    return "N/A"

def get_model_details(api, model_id):
    """Fetches details and returns (model_dict, status_string)."""
    try:
        model_name_only = model_id.split('/')[-1]

        # 1. Filter out by model id if it's in the ignore list
        if model_id in IGNORE_MODELS:
            return None, "Manually ignored"

        # 2. Filter out by MoE Regex Pattern
        if MOE_PATTERN.search(model_name_only):
            return None, "MoE architecture"

        # 3. Filter out by ignored words in the name
        model_id_lower = model_id.lower()
        for word in IGNORE_WORDS:
            if word.lower() in model_id_lower:
                return None, f"Ignored word '{word}'"

        # Fetch model info only after local name checks pass
        info = api.model_info(model_id)

        # 4. Filter out ignored model types
        if info.pipeline_tag and info.pipeline_tag.lower() in IGNORE_TYPES:
            return None, f"Ignored type '{info.pipeline_tag}'"

        if info.tags:
            for tag in info.tags:
                if tag.lower() in IGNORE_TYPES:
                    return None, f"Ignored tag '{tag}'"

        # 5. Extract Release Date
        release_date = info.created_at.strftime("%Y-%m-%d") if info.created_at else "N/A"

        # 6. Total Parameters (Metadata -> Regex Fallback)
        total_params_raw = "N/A"
        if info.safetensors and "total" in info.safetensors:
            total_params_raw = format_to_billions(info.safetensors["total"])
        else:
            total_params_raw = extract_params_from_name(model_id)

        if total_params_raw != "N/A" and float(total_params_raw) > MAX_PARAMS:
            return None, f"Exceeds max params ({total_params_raw}B > {MAX_PARAMS}B)"

        return {
            "model": model_id,
            "total_parameters": total_params_raw,
            "release_date": release_date
        }, "Accepted"
    except Exception as e:
        return None, f"API Error: {e}"


### 3. Fetching Data from Hugging Face
We iterate over the list of chosen collections, fetch the models in each collection, and apply our filtering logic to compile a list of viable models.

In [16]:
COLLECTIONS = [
    "Qwen/qwen",
    "Qwen/qwen15",
    "Qwen/qwen2",
    "Qwen/qwen25",
    "Qwen/qwen3",
    "Qwen/qwen35",
    "meta-llama/llama-2-family",
    "meta-llama/meta-llama-3",
    "meta-llama/llama-31",
    "meta-llama/llama-32",
    "meta-llama/llama-33",
    "meta-llama/llama-4",
    "deepseek-ai/deepseek-llm",
    "deepseek-ai/deepseek-v2",
    "deepseek-ai/deepseek-v25",
    "deepseek-ai/deepseek-v3",
    "deepseek-ai/deepseek-v31",
    "deepseek-ai/deepseek-v32",
    "google/gemma-release",
    "google/gemma-2-release",
    "google/gemma-3-release",
    "google/gemma-3n",
    "google/gemma-4",
    "mistralai/ministral-3"
]

api = HfApi()
rows = []

for col_id in COLLECTIONS:
    print(f"\n📦 Collection: {col_id}")
    try:
        collection = get_collection(col_id)
        for item in collection.items:
            if item.item_type == "model":
                details, reason = get_model_details(api, item.item_id)
                
                if details:
                    details["collection"] = col_id
                    details["collection_name"] = collection.title
                    rows.append(details)
                    print(f"  ✓ {item.item_id}")
                else:
                    print(f"  ✗ {item.item_id} (Reason: {reason})")
    except Exception as e:
        print(f"  [!] Collection Error: {e}")



📦 Collection: Qwen/qwen
  ✗ Qwen/Qwen-72B (Reason: Exceeds max params (72.29B > 50.0B))
  ✗ Qwen/Qwen-72B-Chat (Reason: Exceeds max params (72.29B > 50.0B))
  ✗ Qwen/Qwen-72B-Chat-Int4 (Reason: Ignored word 'int4')
  ✗ Qwen/Qwen-72B-Chat-Int8 (Reason: Ignored word 'int8')
  ✓ Qwen/Qwen-14B
  ✓ Qwen/Qwen-14B-Chat
  ✗ Qwen/Qwen-14B-Chat-Int4 (Reason: Ignored word 'int4')
  ✗ Qwen/Qwen-14B-Chat-Int8 (Reason: Ignored word 'int8')
  ✓ Qwen/Qwen-7B
  ✓ Qwen/Qwen-7B-Chat
  ✗ Qwen/Qwen-7B-Chat-Int4 (Reason: Ignored word 'int4')
  ✗ Qwen/Qwen-7B-Chat-Int8 (Reason: Ignored word 'int8')
  ✓ Qwen/Qwen-1_8B
  ✓ Qwen/Qwen-1_8B-Chat
  ✗ Qwen/Qwen-1_8B-Chat-Int4 (Reason: Ignored word 'int4')
  ✗ Qwen/Qwen-1_8B-Chat-Int8 (Reason: Ignored word 'int8')

📦 Collection: Qwen/qwen15
  ✗ Qwen/Qwen1.5-110B-Chat (Reason: Exceeds max params (111.21B > 50.0B))
  ✗ Qwen/Qwen1.5-110B (Reason: Exceeds max params (111.21B > 50.0B))
  ✓ Qwen/Qwen1.5-32B
  ✓ Qwen/Qwen1.5-32B-Chat
  ✗ Qwen/Qwen1.5-72B-Chat (Reason: Exc

### 4. Post-Processing: Prioritize Instruction-Tuned Models
If a base model and its instruction-tuned variant (e.g., `-it`, `-chat`) both exist in the parsed list, we remove the base model and keep only the instruction-tuned version.

In [32]:
model_map = {r["model"]: r for r in rows}
to_remove = {}

for model_id in model_map.keys():
    for s in PRIORITIZE_SUFFIXES:
        s_lower = s.lower()
        model_id_lower = model_id.lower()
        if model_id_lower.endswith(s_lower):
            base_name = model_id[:-len(s)]
            if base_name in model_map:
                to_remove[base_name] = model_id

print("\n🔄 Prioritizing Models:")
if not to_remove:
    print("  - No base models needed removal.")
else:
    for base, tuned in to_remove.items():
        print(f"  ✗ Removed {base}")
        print(f"  ↳ ✓ Kept {tuned}")

final_rows = [r for r in rows if r["model"] not in to_remove]



🔄 Prioritizing Models:
  ✗ Removed Qwen/Qwen-14B
  ↳ ✓ Kept Qwen/Qwen-14B-Chat
  ✗ Removed Qwen/Qwen-7B
  ↳ ✓ Kept Qwen/Qwen-7B-Chat
  ✗ Removed Qwen/Qwen-1_8B
  ↳ ✓ Kept Qwen/Qwen-1_8B-Chat
  ✗ Removed Qwen/Qwen1.5-32B
  ↳ ✓ Kept Qwen/Qwen1.5-32B-Chat
  ✗ Removed Qwen/Qwen1.5-4B
  ↳ ✓ Kept Qwen/Qwen1.5-4B-Chat
  ✗ Removed Qwen/Qwen1.5-0.5B
  ↳ ✓ Kept Qwen/Qwen1.5-0.5B-Chat
  ✗ Removed Qwen/Qwen1.5-14B
  ↳ ✓ Kept Qwen/Qwen1.5-14B-Chat
  ✗ Removed Qwen/Qwen1.5-1.8B
  ↳ ✓ Kept Qwen/Qwen1.5-1.8B-Chat
  ✗ Removed Qwen/Qwen1.5-7B
  ↳ ✓ Kept Qwen/Qwen1.5-7B-Chat
  ✗ Removed Qwen/Qwen2-7B
  ↳ ✓ Kept Qwen/Qwen2-7B-Instruct
  ✗ Removed Qwen/Qwen2-1.5B
  ↳ ✓ Kept Qwen/Qwen2-1.5B-Instruct
  ✗ Removed Qwen/Qwen2-0.5B
  ↳ ✓ Kept Qwen/Qwen2-0.5B-Instruct
  ✗ Removed Qwen/Qwen2.5-0.5B
  ↳ ✓ Kept Qwen/Qwen2.5-0.5B-Instruct
  ✗ Removed Qwen/Qwen2.5-1.5B
  ↳ ✓ Kept Qwen/Qwen2.5-1.5B-Instruct
  ✗ Removed Qwen/Qwen2.5-3B
  ↳ ✓ Kept Qwen/Qwen2.5-3B-Instruct
  ✗ Removed Qwen/Qwen2.5-7B
  ↳ ✓ Kept Qwen/Qwen

### 5. Build DataFrame
We ingest our final collected row definitions into a structured Pandas DataFrame to facilitate easier data processing.

In [18]:
df = pd.DataFrame(final_rows)
df['total_parameters'] = pd.to_numeric(df['total_parameters'], errors='coerce')
df = df.dropna(subset=['total_parameters']).copy()
column_data = df['total_parameters']
df.head()

,model,total_parameters,release_date,collection,collection_name
0,Qwen/Qwen-14B-Chat,14.17,2023-09-24,Qwen/qwen,Qwen
1,Qwen/Qwen-7B-Chat,7.72,2023-08-03,Qwen/qwen,Qwen
2,Qwen/Qwen-1_8B-Chat,8.00,2023-11-30,Qwen/qwen,Qwen
3,Qwen/Qwen1.5-32B-Chat,32.51,2024-04-03,Qwen/qwen15,Qwen1.5
4,Qwen/Qwen1.5-4B-Chat,3.95,2024-01-30,Qwen/qwen15,Qwen1.5


### 6. Calculate Natural Breaks with Jenks
To properly categorize models into distinct size categories without arbitrary thresholds, we use Jenks Natural Breaks optimization. This seeks to minimize each class's average deviation from the class mean, while maximizing each class's deviation from the means of the other groups.

In [19]:
values = column_data.dropna().astype(float).to_list()
breaks = jenkspy.jenks_breaks(values, n_classes=4)
breaks

[np.float64(0.27),
 np.float64(5.12),
 np.float64(9.65),
 np.float64(15.71),
 np.float64(32.68)]

### 7. Assign Labels
Apply the calculated bins to split our dataframe into 'Tiny/Mobile', 'Small/Edge', 'Standard/Desktop', and 'Mid-range/Server' tiers.

In [20]:
df['category'] = pd.cut(column_data, bins=breaks, labels=["Tiny/Mobile", "Small/Edge", "Standard/Desktop", "Mid-range/Server"], include_lowest=True)
df = df[['collection', 'collection_name', 'model', 'total_parameters', 'release_date', 'category']]
df

,collection,collection_name,model,total_parameters,release_date,category
0,Qwen/qwen,Qwen,Qwen/Qwen-14B-Chat,14.17,2023-09-24,Standard/Desktop
1,Qwen/qwen,Qwen,Qwen/Qwen-7B-Chat,7.72,2023-08-03,Small/Edge
2,Qwen/qwen,Qwen,Qwen/Qwen-1_8B-Chat,8.00,2023-11-30,Small/Edge
3,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-32B-Chat,32.51,2024-04-03,Mid-range/Server
4,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-4B-Chat,3.95,2024-01-30,Tiny/Mobile
5,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-0.5B-Chat,0.62,2024-01-31,Tiny/Mobile
6,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-14B-Chat,14.17,2024-01-30,Standard/Desktop
7,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-1.8B-Chat,1.84,2024-01-30,Tiny/Mobile
8,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-7B-Chat,7.72,2024-01-30,Small/Edge
9,Qwen/qwen2,Qwen2,Qwen/Qwen2-7B-Instruct,7.62,2024-06-04,Small/Edge


### 8. Filter Selection per Family & Size Category

For the final experimental selection:

1. If multiple model sizes exist for the same model family version inside the same category, choose the model with the largest parameter size.
2. In case of a tie in the number of parameters, choose the model with the most recent release date.

We then save the resulting dataset to a CSV file.

In [19]:
df["total_parameters"] = pd.to_numeric(df["total_parameters"], errors="coerce")
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

filtered_df = (
    df.sort_values(
        by=["category", "collection", "total_parameters", "release_date"],
        ascending=[True, True, False, False],
    )
    .drop_duplicates(subset=["category", "collection"], keep="first")
    .copy()
)

filtered_df.sort_values(by=["category", "release_date"], ascending=[True, True], inplace=True)
filtered_df.reset_index(drop=True, inplace=True)

filtered_df.to_csv('selected-models.csv', index=False)

filtered_df

,collection,collection_name,model,total_parameters,release_date,category
0,Qwen/qwen15,Qwen1.5,Qwen/Qwen1.5-4B-Chat,3.95,2024-01-30,Tiny/Mobile
1,google/gemma-release,Gemma release,google/gemma-1.1-2b-it,2.51,2024-03-26,Tiny/Mobile
2,Qwen/qwen2,Qwen2,Qwen/Qwen2-1.5B-Instruct,1.54,2024-06-03,Tiny/Mobile
3,google/gemma-2-release,Gemma 2 Release,google/gemma-2-2b-it,2.61,2024-07-16,Tiny/Mobile
4,Qwen/qwen25,Qwen2.5,Qwen/Qwen2.5-3B-Instruct,3.09,2024-09-17,Tiny/Mobile
5,meta-llama/llama-32,Llama 3.2,meta-llama/Llama-3.2-3B-Instruct,3.21,2024-09-18,Tiny/Mobile
6,google/gemma-3-release,Gemma 3 Release,google/shieldgemma-2-4b-it,4.00,2025-03-04,Tiny/Mobile
7,google/gemma-3n,Gemma 3n,google/gemma-3n-E4B-it,4.00,2025-06-03,Tiny/Mobile
8,Qwen/qwen3,Qwen3,Qwen/Qwen3-4B-Instruct-2507,4.02,2025-08-05,Tiny/Mobile
9,mistralai/ministral-3,Ministral 3,mistralai/Ministral-3-3B-Instruct-2512,3.00,2025-10-31,Tiny/Mobile


### 9. Final Statistics
Displaying the number of models successfully selected per category.

In [ ]:
filtered_df.groupby("category").size().reindex(
    ["Tiny/Mobile", "Small/Edge", "Standard/Desktop", "Mid-range/Server"]
)

category
Tiny/Mobile         12
Small/Edge          14
Standard/Desktop     8
Mid-range/Server     7
dtype: int64